# Factor Portfolio – EU

**Pipeline:** Runs the same script and data loading as `run_all.py` (all factors: Value, Momentum, Liquidity, Quality, Yield/BETA0–BETA2, Low Vol), then builds the EU factor return matrix. No need to run factors or `run_all.py` beforehand.

**Directory structure:**
- Inputs: `src/`, `config.py`, data as required by factor modules.
- Outputs: `outputs/factors/`, `outputs/portfolio_eu/`, **`outputs/plots/eu/`** (all figures from this notebook).

**Parameters:** Weights and settings from `config.py` (`DEFAULT_WEIGHTS_EU`). Run notebook from project root.

## Setup, load pipeline, build factor matrix

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project root (works from project root or from notebooks/)
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

PLOTS_DIR = ROOT / "outputs" / "plots" / "eu"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

from run_all import run_all_factors, build_factor_matrix
import config

factor_dfs = run_all_factors()
factor_returns = build_factor_matrix(factor_dfs, 'eu')
factor_returns = factor_returns.dropna(how='all')

weights = {k.replace('_EU',''): v for k, v in config.DEFAULT_WEIGHTS_EU.items() if k.replace('_EU','') in factor_returns.columns}
weighted_ret = sum(factor_returns[c].fillna(0) * weights.get(c, 0) for c in factor_returns.columns if c in weights)
weighted_ret = pd.Series(weighted_ret, index=factor_returns.index)

print('✓ EU factor matrix:', factor_returns.shape)
display(factor_returns.tail())

## Calculated output metrics (full table)

In [ ]:
def performance_stats(df):
    stats = {}
    for col in df.columns:
        ret = df[col].dropna()
        if len(ret) < 2: continue
        ann_ret = ret.mean() * 12
        ann_vol = ret.std() * np.sqrt(12)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
        cum = (1 + ret).cumprod()
        cum_ret = cum.iloc[-1] - 1
        dd = (cum - cum.cummax()) / cum.cummax()
        max_dd = dd.min()
        win_rate = (ret > 0).mean() * 100
        skew = ret.skew()
        kurt = ret.kurtosis()
        stats[col] = {
            'Ann. Return': ann_ret, 'Ann. Vol': ann_vol, 'Sharpe': sharpe,
            'Cum. Return': cum_ret, 'Max DD': max_dd,
            'Win Rate %': win_rate, 'Skewness': skew, 'Kurtosis': kurt
        }
    return pd.DataFrame(stats).T

perf = performance_stats(factor_returns)
print('Per-factor metrics (EU):')
display(perf.round(4))

# Weighted portfolio metrics
wp_stats = performance_stats(pd.DataFrame({'Weighted (config)': weighted_ret}))
print('\nWeighted portfolio (config weights):')
display(wp_stats.round(4))

## Factor returns over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for col in factor_returns.columns:
    s = factor_returns[col].dropna()
    if len(s) > 0: ax.plot(s.index, s, label=col, alpha=0.8)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_ylabel('Monthly return')
ax.set_title('EU Factor returns over time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'factor_returns_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

## Cumulative returns (1$ invested per factor) & Sharpe bar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cum = (1 + factor_returns).cumprod()
cum.plot(ax=axes[0])
axes[0].set_title('EU Cumulative returns (1$ invested per factor)')
axes[0].set_ylabel('Cumulative return')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.3)

perf['Sharpe'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='navy')
axes[1].set_title('Annualized Sharpe ratio by factor')
axes[1].set_ylabel('Sharpe')
axes[1].axhline(0, color='gray', linestyle='-')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cumulative_and_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

## Drawdown by factor & Rolling 12m Sharpe (weighted portfolio)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
cum = (1 + factor_returns).cumprod()
dd = (cum - cum.cummax()) / cum.cummax()
dd.plot(ax=axes[0])
axes[0].set_title('EU Drawdown by factor')
axes[0].set_ylabel('Drawdown')
axes[0].legend(loc='lower left', fontsize=8)
axes[0].grid(True, alpha=0.3)

roll_ret = weighted_ret.rolling(12).mean() * 12
roll_vol = weighted_ret.rolling(12).std() * np.sqrt(12)
roll_sharpe = (roll_ret / roll_vol.replace(0, np.nan)).dropna()
roll_sharpe.plot(ax=axes[1], color='#A23B72')
axes[1].set_title('Rolling 12-month Sharpe (EU weighted portfolio, config weights)')
axes[1].set_ylabel('Sharpe')
axes[1].axhline(0, color='gray', linestyle='-')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'drawdown_and_rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation heatmap (factor returns)

In [ ]:
corr = factor_returns.corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_title('EU Factor return correlation matrix')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
display(corr.round(3))

## Return distributions (histograms by factor)

In [ ]:
cols = factor_returns.columns.tolist()
n = len(cols)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3 * nrows))
axes = np.atleast_2d(axes)
for idx, col in enumerate(cols):
    r, c = idx // ncols, idx % ncols
    ax = axes[r, c]
    ser = factor_returns[col].dropna()
    ax.hist(ser, bins=25, edgecolor='black', alpha=0.7)
    ax.axvline(ser.mean(), color='red', linestyle='--', label=f'Mean={ser.mean():.3f}')
    ax.set_title(f'{col} (EU)')
    ax.set_xlabel('Monthly return')
    ax.legend(fontsize=8)
for idx in range(len(cols), nrows * ncols):
    r, c = idx // ncols, idx % ncols
    axes[r, c].set_visible(False)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Weighted portfolio cumulative return

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
cum = (1 + weighted_ret).cumprod()
ax.plot(cum.index, cum.values, color='#A23B72', linewidth=2)
ax.set_title('EU weighted portfolio cumulative return (config weights)')
ax.set_ylabel('Cumulative return')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'cumulative_weighted.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ All plots saved to', PLOTS_DIR)